In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from imblearn.over_sampling import SMOTE
from itertools import combinations
import numpy as np

In [22]:
# data_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project .v2\Cancer Prediction v2. - DS1.csv" 
# data_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project .v2\Cancer Prediction v2. - DS2.csv" 
data_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project .v2\Cancer Prediction v2. - DS3.csv" 
# data_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project .v2\Cancer Prediction v2. - DS4.csv" 
# data_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project .v2\Cancer Prediction v2. - DS5.csv" 
df = pd.read_csv(data_path, delimiter=",")

# Features and target
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print(np.bincount(y))
smote = SMOTE(random_state=42, k_neighbors=5)

X, y = smote.fit_resample(X, y)

print(np.bincount(y))
# X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# X_train, X_cv, y_train, y_cv = train_test_split(X_temp, y_temp, test_size=1/8, random_state=42, stratify=y_temp)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(df)

[624 155]
[624 624]
     201270_x_at  207033_at  208772_at  bone.relapses..1.yes..0.no..ch1
0      -0.281828  -0.368418   1.045710                                0
1       0.828244   1.169221  -0.701383                                0
2       0.016777   0.899299   0.216254                                0
3       1.023898   0.862940  -1.053899                                0
4      -1.365721   0.799499  -1.030472                                0
..           ...        ...        ...                              ...
774     0.563580  -1.560599  -0.755617                                0
775    -0.214831  -0.106444  -1.719248                                0
776     0.679457  -0.021725   0.548051                                0
777     0.024110  -0.323976  -1.407429                                0
778     1.908103  -0.201965   0.530323                                1

[779 rows x 4 columns]


In [20]:
import pandas as pd
from itertools import combinations
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, recall_score, precision_score,
    f1_score, accuracy_score
)

gene_names = X.columns.tolist()
results = []

all_results = []
n_classes = len(set(y_train))

for k in range(1, len(gene_names) + 1):
    for comb in combinations(gene_names, k):
        features = list(comb)

        X_train_sel = X_train[features]
        X_cv_sel = X_cv[features]
        X_test_sel = X_test[features]

        # model = LogisticRegression(max_iter=1000)
        model = RandomForestClassifier(n_estimators=310, random_state=42)
        model.fit(X_train_sel, y_train)

        # Predictions
        y_train_pred = model.predict(X_train_sel)
        y_cv_pred = model.predict(X_cv_sel)
        y_test_pred = model.predict(X_test_sel)

        # Probabilities
        y_train_proba = model.predict_proba(X_train_sel)
        y_cv_proba = model.predict_proba(X_cv_sel)
        y_test_proba = model.predict_proba(X_test_sel)

        # AUC scores
        try:
            if n_classes > 2:
                train_auc = roc_auc_score(y_train, y_train_proba, multi_class='ovr', average='macro')
                cv_auc = roc_auc_score(y_cv, y_cv_proba, multi_class='ovr', average='macro')
                test_auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro')
            else:
                train_auc = roc_auc_score(y_train, y_train_proba[:, 1])
                cv_auc = roc_auc_score(y_cv, y_cv_proba[:, 1])
                test_auc = roc_auc_score(y_test, y_test_proba[:, 1])
        except:
            train_auc = cv_auc = test_auc = float('-inf')

        # Accuracy
        train_acc = accuracy_score(y_train, y_train_pred)
        cv_acc = accuracy_score(y_cv, y_cv_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        # Other metrics on TEST
        recall = recall_score(y_test, y_test_pred, average='macro', zero_division=0)
        precision = precision_score(y_test, y_test_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_test_pred, average='macro', zero_division=0)

        # print(f"Genes: {' + '.join(features)}")
        # print(f"Train AUC: {train_auc:.4f}, CV AUC: {cv_auc:.4f}, Test AUC: {test_auc:.4f}")
        # print(f"Train Acc: {train_acc:.4f}, CV Acc: {cv_acc:.4f}, Test Acc: {test_acc:.4f}\n")

        all_results.append({
            "genes": ' + '.join(features),
            "train_auc": train_auc,
            "cv_auc": cv_auc,
            "test_auc": test_auc,
            "recall": recall,
            "precision": precision,
            "f1": f1,
            "train_acc": train_acc,
            "cv_acc": cv_acc,
            "test_acc": test_acc
        })

# Save results
results_df = pd.DataFrame(all_results)

# Summary of best performing combinations
summary_rows = []
metrics = {
    "Highest Train AUC": "train_auc",
    "Highest CV AUC": "cv_auc",
    "Highest Test AUC": "test_auc",
    # "Highest Recall": "recall",
    # "Highest Precision": "precision",
    # "Highest F1-score": "f1",
    "Highest Train Accuracy": "train_acc",
    "Highest CV Accuracy": "cv_acc",
    "Highest Test Accuracy": "test_acc"
}

for label, metric in metrics.items():
    best_row = results_df.loc[results_df[metric].idxmax()]
    summary_rows.append({
        "Metric": label,
        "Genes": best_row["genes"],
        "Train AUC": best_row["train_auc"],
        "CV AUC": best_row["cv_auc"],
        "Test AUC": best_row["test_auc"],
        # "Recall": best_row["recall"],
        # "Precision": best_row["precision"],
        # "F1-score": best_row["f1"],
        "Train Accuracy": best_row["train_acc"],
        "CV Accuracy": best_row["cv_acc"],
        "Test Accuracy": best_row["test_acc"]
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("Results.csv", index=False)

In [ ]:
# import pandas as pd
# from itertools import combinations
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import (
#     roc_auc_score, recall_score, precision_score,
#     f1_score, accuracy_score
# )

# gene_names = X_train.columns.tolist()
# n_classes = len(set(y_train))
# n_estimators_list = [300, 310, 320, 330, 340, 350, 360, 370, 380, 390]  # You can customize this list

# for n_estimators in n_estimators_list:
#     print(f"\n========== Evaluating n_estimators = {n_estimators} ==========\n")
    
#     all_results = []
    
#     for k in range(1, len(gene_names) + 1):
#         for comb in combinations(gene_names, k):
#             features = list(comb)

#             X_train_sel = X_train[features]
#             X_cv_sel = X_cv[features]
#             X_test_sel = X_test[features]

#             model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
#             model.fit(X_train_sel, y_train)

#             # Predictions
#             y_train_pred = model.predict(X_train_sel)
#             y_cv_pred = model.predict(X_cv_sel)
#             y_test_pred = model.predict(X_test_sel)

#             # Probabilities
#             y_train_proba = model.predict_proba(X_train_sel)
#             y_cv_proba = model.predict_proba(X_cv_sel)
#             y_test_proba = model.predict_proba(X_test_sel)

#             # AUC scores
#             try:
#                 if n_classes > 2:
#                     train_auc = roc_auc_score(y_train, y_train_proba, multi_class='ovr', average='macro')
#                     cv_auc = roc_auc_score(y_cv, y_cv_proba, multi_class='ovr', average='macro')
#                     test_auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro')
#                 else:
#                     train_auc = roc_auc_score(y_train, y_train_proba[:, 1])
#                     cv_auc = roc_auc_score(y_cv, y_cv_proba[:, 1])
#                     test_auc = roc_auc_score(y_test, y_test_proba[:, 1])
#             except:
#                 train_auc = cv_auc = test_auc = float('-inf')

#             # Accuracy
#             train_acc = accuracy_score(y_train, y_train_pred)
#             cv_acc = accuracy_score(y_cv, y_cv_pred)
#             test_acc = accuracy_score(y_test, y_test_pred)

#             # Other metrics on TEST
#             recall = recall_score(y_test, y_test_pred, average='macro', zero_division=0)
#             precision = precision_score(y_test, y_test_pred, average='macro', zero_division=0)
#             f1 = f1_score(y_test, y_test_pred, average='macro', zero_division=0)

#             all_results.append({
#                 "genes": ' + '.join(features),
#                 "train_auc": train_auc,
#                 "cv_auc": cv_auc,
#                 "test_auc": test_auc,
#                 "recall": recall,
#                 "precision": precision,
#                 "f1": f1,
#                 "train_acc": train_acc,
#                 "cv_acc": cv_acc,
#                 "test_acc": test_acc
#             })

#     # Save detailed results
#     results_df = pd.DataFrame(all_results)
#     results_df.to_csv(f"Results_n{n_estimators}.csv", index=False)

#     # Summary of best performing combinations
#     summary_rows = []
#     metrics = {
#         "Highest Train AUC": "train_auc",
#         "Highest CV AUC": "cv_auc",
#         "Highest Test AUC": "test_auc",
#         "Highest Train Accuracy": "train_acc",
#         "Highest CV Accuracy": "cv_acc",
#         "Highest Test Accuracy": "test_acc"
#     }

#     for label, metric in metrics.items():
#         best_row = results_df.loc[results_df[metric].idxmax()]
#         summary_rows.append({
#             "Metric": label,
#             "Genes": best_row["genes"],
#             "Train AUC": best_row["train_auc"],
#             "CV AUC": best_row["cv_auc"],
#             "Test AUC": best_row["test_auc"],
#             "Train Accuracy": best_row["train_acc"],
#             "CV Accuracy": best_row["cv_acc"],
#             "Test Accuracy": best_row["test_acc"]
#         })

#         print(f"{label}")
#         print(f"  Genes: {best_row['genes']}")
#         print(f"  Train AUC: {best_row['train_auc']:.4f}, CV AUC: {best_row['cv_auc']:.4f}, Test AUC: {best_row['test_auc']:.4f}")
#         print(f"  Train Acc: {best_row['train_acc']:.4f}, CV Acc: {best_row['cv_acc']:.4f}, Test Acc: {best_row['test_acc']:.4f}")
#         print("")

#     # Save summary
#     # summary_df = pd.DataFrame(summary_rows)
#     # summary_df.to_csv(f"Summary_n{n_estimators}.csv", index=False)



========== Evaluating n_estimators = 300 ==========

Highest Train AUC
  Genes: 201270_x_at
  Train AUC: 1.0000, CV AUC: 0.6359, Test AUC: 0.6460
  Train Acc: 1.0000, CV Acc: 0.6160, Test Acc: 0.5880

Highest CV AUC
  Genes: 201270_x_at + 207033_at + 208772_at
  Train AUC: 1.0000, CV AUC: 0.8911, Test AUC: 0.8400
  Train Acc: 1.0000, CV Acc: 0.8000, Test Acc: 0.7800

Highest Test AUC
  Genes: 201270_x_at + 207033_at + 208772_at
  Train AUC: 1.0000, CV AUC: 0.8911, Test AUC: 0.8400
  Train Acc: 1.0000, CV Acc: 0.8000, Test Acc: 0.7800

Highest Train Accuracy
  Genes: 201270_x_at
  Train AUC: 1.0000, CV AUC: 0.6359, Test AUC: 0.6460
  Train Acc: 1.0000, CV Acc: 0.6160, Test Acc: 0.5880

Highest CV Accuracy
  Genes: 201270_x_at + 207033_at + 208772_at
  Train AUC: 1.0000, CV AUC: 0.8911, Test AUC: 0.8400
  Train Acc: 1.0000, CV Acc: 0.8000, Test Acc: 0.7800

Highest Test Accuracy
  Genes: 201270_x_at + 207033_at + 208772_at
  Train AUC: 1.0000, CV AUC: 0.8911, Test AUC: 0.8400
  Train Ac

KeyboardInterrupt: 

In [25]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# CONFIGURATION
k_folds = 5
random_state = 42

# X_train and y_train are assumed to already be loaded
gene_names = X_train.columns.tolist()

best_combination = None
best_cv_score = -np.inf
all_results = []

# Evaluate combinations of 1 to all genes
for k in range(1, len(gene_names) + 1):
    for comb in combinations(gene_names, k):
        features = list(comb)
        X_subset = X_train[features]

        # model = LogisticRegression(max_iter=1000) 
        model = RandomForestClassifier(n_estimators=310, random_state=42)

        # Cross-validation scoring by accuracy
        cv_scores = cross_val_score(model, X_subset, y_train, cv=k_folds, scoring='accuracy')
        avg_score = np.mean(cv_scores)

        all_results.append({
            "Gene Combination": ' + '.join(features),
            "CV Accuracy": round(avg_score, 4)
        })

        # Save best
        if avg_score > best_cv_score:
            best_cv_score = avg_score
            best_combination = features

# Save all results
results_df = pd.DataFrame(all_results)
results_df.sort_values(by="CV Accuracy", ascending=False, inplace=True)
# results_df.to_csv("Results.csv", index=False)

# Print best
print(best_combination, f"{best_cv_score:.8f}")


['201270_x_at', '207033_at', '208772_at'] 0.73195402
